# Single-Step Extraction Pipeline — GPT-5.4, 210 abstracts (GT0513)

One-prompt extraction using the full system prompt. Outputs a CSV compatible with `graph_structural_eval_parallel.ipynb`.

**Output columns:** `article_id, title, abstract, status, error, variables, variable_count, step5_1_hierarchy, hierarchy_count, step5_1_directional, directional_count, step5_1_correlational, correlational_count, step5_1_moderation, moderation_count, justifications, input_tokens, output_tokens, total_tokens`

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('OpenAI client ready.')

In [ ]:
# ---- Config ----
ROOT       = Path('..').resolve() if Path('..').joinpath('data').exists() else Path('.').resolve()
GT_JSON    = ROOT / 'data' / 'raw' / 'EmpiriGraph-Psy_gold_annotation.json'
OUTPUT_DIR = ROOT / 'data' / 'output'
OUTPUT_CSV = OUTPUT_DIR / 'single_step_gpt54_210_results.csv'
BATCH_DIR  = OUTPUT_DIR / 'single_step_gpt54_210_batch'
BATCH_DIR.mkdir(parents=True, exist_ok=True)

MODEL      = 'gpt-5.4'
MAX_TOKENS = 4096

print(f'GT_JSON:    {GT_JSON}')
print(f'OUTPUT_CSV: {OUTPUT_CSV}')
print(f'MODEL:      {MODEL}')


In [ ]:
# ---- Abstract cleaning ----
_CLEAN_RE = re.compile(
    r'\s*\(PsycInfo Database Record.*$'
    r'|\s*\(PsycINFO Database Record.*$',
    re.DOTALL | re.IGNORECASE,
)
_IMPACT_RE = re.compile(
    r'\s*(Impact Statement|Public Significance Statement|Public Significance|'
    r'Practitioner Points|Educational Impact and Implications Statement|'
    r'Translational Significance|Clinical Impact|Lay Summary|Public Interest Summary|'
    r'What is the public significance|What does this article add)'
    r'[:\-\s].*$',
    re.DOTALL | re.IGNORECASE,
)

def clean_abstract(raw: str) -> str:
    text = _CLEAN_RE.sub('', raw).strip()
    text = _IMPACT_RE.sub('', text).strip()
    return text

print('clean_abstract defined.')

In [ ]:
# ---- Prompt ----
SYSTEM_PROMPT = """You are an expert scientific-text analyst specializing in variable detection, hierarchy normalization, relevant sentence extraction, relationship extraction, and relationship validation in research abstracts."""

USER_PROMPT_TEMPLATE = """<task>
Your task is to extract and identify relationships between variables from the abstract provided. 
</task>
<definitions>
Extract only true study variables explicitly investigated, measured, categorized, manipulated, or reported as substantive constructs in the abstract.
RELATIONSHIP TYPES
- DIRECTIONAL (Var1 -> Var2): explicit direction/influence/prediction/causal or predictor-outcome claim.
- CORRELATIONAL (Var1 <-> Var2): association without directional claim.
- MODERATION: third variable modifies relationship between two (or path-level set) variables.
- Moderation must also include underlying relationship(s).
- Interaction effects produce both directional and moderation entries.
HIERARCHY: VALID CASES
CASE 1: Construct decomposition
- Abstract identifies/distinguishes/analyzes components/subthemes of broader concept, and this decomposition is a main reported finding (often qualitative/mixed methods).
- Not valid if components are only definitional/background or simple appositional listing without reported finding status.
CASE 2: Abstract-to-specific relationship
- Broad construct is explicitly mentioned, then lower-level dimensions/indicators are examined in relation to other variables.
</definitions>


<input>
ABSTRACT:
{abstract}
</input>
<output_format>
Return a JSON object with this exact structure:
{{
  "hierarchy": {{
    "hierarchical_relationships": [
      {{
        "higher_level_variable": "",
        "lower_level_variable": ""
      }}
    ]
  }},
  "final_variable_list": ["", ""],
  "independent_variables": [
    {{
      "variable": "",
      "components": ["", ""]
    }}
  ],
  "dependent_variables": [
    {{
      "variable": "",
      "components": ["", ""]
    }}
  ],
  "moderators": ["", ""],
  "mediators": ["", ""],
  "other_variables_controls": [
    {{
      "name": "",
      "role": ""
    }}
  ],
  "relevant_sentences": [
    "Exact sentence 1 from abstract.",
    "Exact sentence 2 from abstract."
  ],
  "directional": [
    ["source_var", "direction", "target_var", "validated|null|hypothesized"]
  ],
  "correlational": [
    ["var1", "correlation", "var2", "validated|null|hypothesized"]
  ],
  "moderation": [
    {{
      "moderator": "moderator_var",
      "moderated_variables": ["var1", "var2"],
      "validation": "validated|null|hypothesized"
    }}
  ],
  "justifications": [
    {{
      "change": "description",
      "reason": "based on abstract"
    }}
  ]
}}
</output_format>
<strict_output_requirements>
- Return ONLY the JSON object, no additional text.
- Use exact variable names from abstract/final normalized list.
- relevant_sentences must be exact spans from abstract; no paraphrasing.
- If a field has no entries, return an empty list.
</strict_output_requirements>"""

print('Prompt template defined.')

In [ ]:
# ---- Load GT articles ----
with open(GT_JSON, 'r', encoding='utf-8') as f:
    gt_articles = json.load(f)

articles = []
for art in gt_articles:
    aid  = int(art['id'])
    data = art.get('data', {})
    raw_ab = data.get('AB', '') or ''
    title  = data.get('TI', '') or ''
    ab     = clean_abstract(raw_ab)
    if ab.strip():
        articles.append({'article_id': f'json_{aid}', 'title': title, 'abstract': ab})

print(f'Loaded {len(articles)} articles with non-empty abstracts.')

In [ ]:
# ---- Submit batch ----
BATCH_INPUT = BATCH_DIR / 'batch_input.jsonl'

with open(BATCH_INPUT, 'w', encoding='utf-8') as f:
    for art in articles:
        user_msg = USER_PROMPT_TEMPLATE.format(abstract=art['abstract'])
        req = {
            'custom_id': art['article_id'],
            'method': 'POST',
            'url': '/v1/chat/completions',
            'body': {
                'model': MODEL,
                'max_completion_tokens': MAX_TOKENS,
                'messages': [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_msg},
                ],
            }
        }
        f.write(json.dumps(req) + '\n')

print(f'Written {len(articles)} requests to {BATCH_INPUT}')

# Upload and submit
with open(BATCH_INPUT, 'rb') as f:
    batch_file = client.files.create(file=f, purpose='batch')
print(f'Uploaded file: {batch_file.id}')

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint='/v1/chat/completions',
    completion_window='24h',
)
print(f'Batch submitted: {batch.id}  status={batch.status}')

# Save batch ID
(BATCH_DIR / 'batch_id.txt').write_text(batch.id)
print(f'Batch ID saved to {BATCH_DIR}/batch_id.txt')

In [ ]:
# ---- Check batch status (run repeatedly until completed) ----
batch_id = (BATCH_DIR / 'batch_id.txt').read_text().strip()
batch = client.batches.retrieve(batch_id)
counts = batch.request_counts
print(f'batch_id : {batch_id}')
print(f'status   : {batch.status}')
print(f'completed: {counts.completed}/{counts.total}  failed={counts.failed}')
if batch.output_file_id:
    print(f'output_file_id: {batch.output_file_id}')
    (BATCH_DIR / 'output_file_id.txt').write_text(batch.output_file_id)
    print('Saved output_file_id.txt — proceed to download cell.')
if batch.error_file_id:
    print(f'error_file_id:  {batch.error_file_id}')


In [ ]:
# ---- Download results ----
output_file_id = (BATCH_DIR / 'output_file_id.txt').read_text().strip()
raw_bytes = client.files.content(output_file_id).content
OUTPUT_JSONL = BATCH_DIR / 'batch_output.jsonl'
OUTPUT_JSONL.write_bytes(raw_bytes)
print(f'Downloaded {len(raw_bytes)} bytes -> {OUTPUT_JSONL}')

# Parse into dict keyed by custom_id
responses = {}
for line in OUTPUT_JSONL.read_text(encoding='utf-8').splitlines():
    if not line.strip(): continue
    obj = json.loads(line)
    cid = obj['custom_id']
    if obj.get('error'):
        responses[cid] = {'error': str(obj['error']), 'content': None, 'usage': None}
    else:
        choice  = obj['response']['body']['choices'][0]
        content = choice['message']['content']
        usage   = obj['response']['body'].get('usage', {})
        responses[cid] = {'error': None, 'content': content, 'usage': usage}

print(f'Parsed {len(responses)} responses.')

In [ ]:
# ---- Parse JSON and build CSV ----

def safe_parse_json(text):
    """Strip markdown fences and parse JSON."""
    if text is None:
        return None, 'null content'
    t = text.strip()
    t = re.sub(r'^```(?:json)?\s*', '', t)
    t = re.sub(r'\s*```$', '', t)
    try:
        return json.loads(t), None
    except Exception as e:
        return None, str(e)

def fmt_hierarchy(parsed):
    """Format hierarchy list as semicolon-separated 'A -> B [CASE 2]' strings."""
    if not parsed: return ''
    h = parsed.get('hierarchy', {})
    rels = h.get('hierarchical_relationships', []) if isinstance(h, dict) else []
    parts = []
    for r in rels:
        hi = r.get('higher_level_variable', '').strip()
        lo = r.get('lower_level_variable', '').strip()
        if hi and lo:
            parts.append(f'{hi} -> {lo} [CASE 2]')
    return '; '.join(parts)

def fmt_directional(parsed):
    if not parsed: return ''
    parts = []
    for item in parsed.get('directional', []):
        if len(item) >= 4:
            src, _, tgt, val = item[0], item[1], item[2], item[3]
            parts.append(f'{src} -> {tgt} [{val}]')
        elif len(item) == 3:
            parts.append(f'{item[0]} -> {item[2]} [hypothesized]')
    return '; '.join(parts)

def fmt_correlational(parsed):
    if not parsed: return ''
    parts = []
    for item in parsed.get('correlational', []):
        if len(item) >= 4:
            parts.append(f'{item[0]} <-> {item[2]} [{item[3]}]')
        elif len(item) == 3:
            parts.append(f'{item[0]} <-> {item[2]} [hypothesized]')
    return '; '.join(parts)

def fmt_moderation(parsed):
    if not parsed: return ''
    parts = []
    for item in parsed.get('moderation', []):
        mod = item.get('moderator', '')
        mvars = item.get('moderated_variables', [])
        val   = item.get('validation', 'hypothesized')
        if mod and len(mvars) >= 2:
            parts.append(f'{mvars[0]} -> {mvars[1]} [moderated by {mod}] [{val}]')
    return '; '.join(parts)

def fmt_justifications(parsed):
    if not parsed: return ''
    parts = []
    for j in parsed.get('justifications', []):
        c = j.get('change', ''); r = j.get('reason', '')
        parts.append(f'{c}: {r}' if r else c)
    return ' | '.join(parts)

# Build article lookup
art_lookup = {a['article_id']: a for a in articles}

rows = []
for art in articles:
    aid  = art['article_id']
    resp = responses.get(aid, {})
    err  = resp.get('error')
    content = resp.get('content')
    usage   = resp.get('usage') or {}

    parsed, parse_err = safe_parse_json(content)
    if parse_err and not err:
        err = f'json_parse: {parse_err}'

    hier_str  = fmt_hierarchy(parsed)
    dir_str   = fmt_directional(parsed)
    corr_str  = fmt_correlational(parsed)
    mod_str   = fmt_moderation(parsed)
    vars_list = parsed.get('final_variable_list', []) if parsed else []
    just_str  = fmt_justifications(parsed)

    rows.append({
        'article_id':           aid,
        'title':                art['title'],
        'abstract':             art['abstract'],
        'status':               'error' if err else 'success',
        'error':                err or '',
        'variables':            '; '.join(vars_list),
        'variable_count':       len(vars_list),
        'step5_1_hierarchy':    hier_str,
        'hierarchy_count':      len(hier_str.split(';')) if hier_str else 0,
        'step5_1_directional':  dir_str,
        'directional_count':    len(dir_str.split(';')) if dir_str else 0,
        'step5_1_correlational':corr_str,
        'correlational_count':  len(corr_str.split(';')) if corr_str else 0,
        'step5_1_moderation':   mod_str,
        'moderation_count':     len(mod_str.split(';')) if mod_str else 0,
        'justifications':       just_str,
        'input_tokens':         usage.get('prompt_tokens', ''),
        'output_tokens':        usage.get('completion_tokens', ''),
        'total_tokens':         usage.get('total_tokens', ''),
    })

result_df = pd.DataFrame(rows)
result_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(result_df)} rows -> {OUTPUT_CSV}')

ok  = (result_df['status'] == 'success').sum()
err = (result_df['status'] == 'error').sum()
print(f'Success: {ok}  Error: {err}')
print(f'\nToken usage:')
print(f'  Input:  {pd.to_numeric(result_df["input_tokens"], errors="coerce").sum():.0f}')
print(f'  Output: {pd.to_numeric(result_df["output_tokens"], errors="coerce").sum():.0f}')
print(f'  Total:  {pd.to_numeric(result_df["total_tokens"], errors="coerce").sum():.0f}')
result_df.head(3)